# 1. 무엇을 진단하나요?

특정 게임의 0ms 기록이 **어떤 패턴으로 남았는지** 확인합니다. 실제 키 누름 시각·입력 성공·묶음 처리의 원인을 확정하는 실험은 아닙니다.
대상 게임과 같은 mode/difficulty의 Train 게임 최대 3개를 비교합니다. 비교 게임은 0ms 비율이 아니라 gameId 순서로 고릅니다.
원본·기존 dataset·split은 읽기 전용이고 모델 fit·재추출·자동 제외·report 덮어쓰기는 하지 않습니다.
**이 Notebook은 작성만 했으며 아직 실행 검증하지 않았습니다.** 위에서 아래로 읽고 실행 결과를 검토하세요.


# 2. 설정과 재사용 코드

기본 대상은 기존 dataset에서 0ms가 집중된 `119e01c9-b982-4907-9480-2308f4604635`입니다.
같은 dataset 안의 다른 게임을 보려면 아래 `TARGET_GAME_ID`만 바꿉니다.
다른 dataset은 이 진단의 입력 계약을 검토한 뒤 새 manifest SHA를 지정해야 합니다.
기존 readiness Notebook의 정의 셀만 재사용하며 실제 감사 실행 셀은 실행하지 않습니다.


In [ ]:
import hashlib
import json
from pathlib import Path

ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
DATASET_RELATIVE = "data/datasets/human-timing-v1-20261024-full"
EXPECTED_MANIFEST_SHA = "c203a1db7b8f2aa7014c565b371088a9614f6abb0df5632e93fe6c807712fc88"
TARGET_GAME_ID = "119e01c9-b982-4907-9480-2308f4604635"
QUALITY_RELATIVE = "data/ml-experiments/human-game-quality-audit/full-report.json"
DEPENDENCY = ROOT / "scripts/ml/notebooks/human_action_data_readiness.ipynb"
shared = {"__name__": "zero_ms_shared"}
for cell in json.loads(DEPENDENCY.read_text(encoding="utf-8"))["cells"]:
    if cell["cell_type"] == "code" and "definitions" in cell["metadata"].get("tags", []):
        exec(compile("".join(cell["source"]), str(DEPENDENCY), "exec"), shared)
require, finite, integer = (shared[name] for name in ("require", "finite", "integer"))
sha256_file, iter_rows = (shared[name] for name in ("sha256_file", "iter_rows"))


def read_only_path(relative):
    relative = Path(relative)
    require(not relative.is_absolute() and ".." not in relative.parts, "INPUT_MUST_BE_REPO_RELATIVE")
    path = (ROOT / relative).resolve(strict=True)
    require(path.is_relative_to(ROOT) and path.is_file(), "INPUT_OUTSIDE_REPOSITORY")
    return path


def select_games(manifest, quality, target):
    require(target in manifest["cohort"]["selectedGameIds"], "TARGET_NOT_IN_DATASET")
    games = {g["gameId"]: g for g in quality["games"]}
    require(target in games, "TARGET_NOT_IN_QUALITY_REPORT")
    context = games[target]
    comparisons = [game for game in sorted(manifest["split"]["gameIds"]["train"])
                   if game != target and game in games
                   and (games[game]["mode"], games[game]["difficulty"]) == (context["mode"], context["difficulty"])][:3]
    return [target, *comparisons]


def diagnose_segment(payload, meta, player_number, lineage=()):
    inputs = payload.get("inputSequence")
    items = inputs if isinstance(inputs, list) else []
    records, valid_times, signatures = [], [], []
    zero_run = max_run = 0
    timing = shared["inspect_segment"](payload, meta, lineage)["readiness"]["logged_inter_action_timing"]
    for index, raw in enumerate(items):
        item = raw if isinstance(raw, dict) else {}
        previous = items[index - 1] if index and isinstance(items[index - 1], dict) else {}
        timestamp = item.get("createdAt")
        previous_time = previous.get("createdAt")
        time_valid = finite(timestamp) and timestamp >= 0
        previous_time_valid = finite(previous_time) and previous_time >= 0
        delta = timestamp - previous_time if index and time_valid and previous_time_valid else None
        if delta is not None and not finite(delta):
            delta = None
        seq_step = item["sequence"] - previous["sequence"] if integer(item.get("sequence")) and integer(previous.get("sequence")) else None
        tick_step = item["tick"] - previous["tick"] if integer(item.get("tick")) and integer(previous.get("tick")) else None
        zero_run = zero_run + 1 if delta == 0 else 0
        max_run = max(max_run, zero_run)
        if time_valid:
            valid_times.append(timestamp)
        action = item.get("action") if isinstance(item.get("action"), str) and item["action"] in shared["ACTIONS"] else "invalid_or_missing"
        if action != "invalid_or_missing" and time_valid and all(integer(item.get(key)) for key in ("sequence", "tick", "pieceIndex")):
            signatures.append((action, item["sequence"], item["tick"], item["pieceIndex"], timestamp))
        records.append({"gameId": meta["gameId"], "playerNumber": player_number,
            "pieceIndex": meta["pieceIndex"], "sourceLine": meta["line"], "inputIndex": index,
            "action": action, "sequence": item.get("sequence") if integer(item.get("sequence")) else None,
            "tick": item.get("tick") if integer(item.get("tick")) else None,
            "timestampMs": timestamp if time_valid else None, "recordedDeltaMs": delta,
            "sequenceStep": seq_step, "tickStep": tick_step,
            "pieceMatches": integer(item.get("pieceIndex")) and item["pieceIndex"] == meta["pieceIndex"],
            "zeroRunPairs": zero_run, "timingStatus": timing})
    placement_time = payload.get("createdAt")
    summary = {"gameId": meta["gameId"], "playerNumber": player_number,
        "pieceIndex": meta["pieceIndex"], "sourceLine": meta["line"], "inputRecords": len(items),
        "invalidSequenceContainer": not isinstance(inputs, list), "timingStatus": timing,
        "hasHold": any(isinstance(item, dict) and item.get("action") == "hold" for item in items),
        "validTimedPairs": sum(r["recordedDeltaMs"] is not None for r in records),
        "zeroPairs": sum(r["recordedDeltaMs"] == 0 for r in records),
        "timeReversals": sum(r["recordedDeltaMs"] is not None and r["recordedDeltaMs"] < 0 for r in records),
        "sequenceProblems": sum(r["inputIndex"] > 0 and r["sequenceStep"] != 1 for r in records),
        "tickProblems": sum(r["inputIndex"] > 0 and (r["tickStep"] is None or r["tickStep"] < 0) for r in records),
        "zeroWithTickAdvance": sum(r["recordedDeltaMs"] == 0 and r["tickStep"] is not None and r["tickStep"] > 0 for r in records),
        "maxZeroRunPairs": max_run,
        "allInputTimesEqual": len(items) >= 2 and len(valid_times) == len(items) and len(set(valid_times)) == 1,
        "inputsEqualPlacementTime": sum(t == placement_time for t in valid_times) if finite(placement_time) else None,
        "repeatedRecordedInputFields": len(signatures) - len(set(signatures))}
    return records, summary


# 3. 작은 예제로 표의 뜻 확인하기

아래는 synthetic 입력입니다. 이동·회전이 같은 시각에 기록됐지만 sequence와 tick은 증가합니다.
이것만으로 두 키를 동시에 눌렀다거나 runtime batch가 원인이라고 확정할 수 없습니다.
이 셀의 assert는 사용자가 실행할 때 작은 계산 실수를 먼저 확인하는 장치이며 실제 게임 진단 결과가 아닙니다.


In [ ]:
example = {"inputSequence": [
    {"action": "move_left", "pieceIndex": 1, "sequence": 1, "tick": 10, "createdAt": 1000},
    {"action": "rotate", "pieceIndex": 1, "sequence": 2, "tick": 11, "createdAt": 1000},
    {"action": "hard_drop", "pieceIndex": 1, "sequence": 3, "tick": 12, "createdAt": 1120},
]}
example_rows, example_summary = diagnose_segment(example, {"gameId": "synthetic", "pieceIndex": 1, "line": 1}, 1)
assert [r["recordedDeltaMs"] for r in example_rows] == [None, 0, 120]
assert example_summary["zeroPairs"] == example_summary["zeroWithTickAdvance"] == 1
assert example_summary["maxZeroRunPairs"] == 1
print("Synthetic only:", example_summary)


# 4. 어떤 원본을 어떻게 읽나요?

Manifest·품질 report·dataset 파일별 SHA와 원본 SHA/bytes를 확인합니다.
기존 streaming parser의 identity 중복/충돌 처리를 재사용하고 선택한 게임의 **전체 Human segment**를 읽습니다.
이는 training에 채택된 pair만 보는 것과 다릅니다. Hold·시간 불충분 구간도 진단에서 표시하되 학습 데이터로 복원하지 않습니다.
이 단계는 원본 전체 metadata scan과 선택 게임 추출 때문에 수 초 이상 걸릴 수 있습니다.
닉네임·원본 playerId·보드·절대 timestamp를 출력에 넣지 않습니다.


In [ ]:
def collect_diagnostic(target=TARGET_GAME_ID):
    import pandas as pd

    manifest_path = read_only_path(DATASET_RELATIVE + "/manifest.json")
    require(sha256_file(manifest_path) == EXPECTED_MANIFEST_SHA, "MANIFEST_SHA_MISMATCH")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"), parse_constant=shared["reject_constant"])
    require(manifest.get("runMode") == "full" and manifest.get("fullCohortCovered") is True
            and manifest.get("experimentId") == "human-timing-dataset", "FULL_TIMING_DATASET_REQUIRED")
    quality_path = read_only_path(QUALITY_RELATIVE)
    require(sha256_file(quality_path) == manifest["qualityPolicy"]["reportSha256"], "QUALITY_SHA_MISMATCH")
    quality = json.loads(quality_path.read_text(encoding="utf-8"), parse_constant=shared["reject_constant"])
    selected = select_games(manifest, quality, target)
    source = read_only_path(manifest["source"]["path"])
    require(source.stat().st_size == manifest["source"]["bytes"]
            and sha256_file(source) == manifest["source"]["sha256"], "SOURCE_SHA_OR_SIZE_MISMATCH")
    require(sha256_file(DEPENDENCY) == manifest["code"]["dependencyNotebookSha256"], "DEPENDENCY_SHA_MISMATCH")
    expected_files = {"train.jsonl", "validation.jsonl", "test.jsonl", "rejected-segments.jsonl", "rejected-pairs.jsonl"}
    require(set(manifest["files"]) == expected_files, "DATASET_FILES_MISMATCH")
    watched = {manifest_path: EXPECTED_MANIFEST_SHA, quality_path: manifest["qualityPolicy"]["reportSha256"],
               source: manifest["source"]["sha256"], DEPENDENCY: manifest["code"]["dependencyNotebookSha256"]}
    used = {}
    for name, info in manifest["files"].items():
        path = read_only_path(DATASET_RELATIVE + "/" + name)
        require(path.stat().st_size == info["bytes"] and sha256_file(path) == info["sha256"], "DATASET_FILE_SHA_MISMATCH")
        watched[path] = info["sha256"]
        if name not in ("train.jsonl", "validation.jsonl", "test.jsonl"):
            continue
        for _, row, kind in iter_rows(path):
            require(kind == "parsed", "DATASET_ROW_INVALID")
            if row["gameId"] in selected:
                key = (row["sourceLine"], row["currentInputIndex"])
                require(key not in used, "DUPLICATE_DATASET_PAIR")
                used[key] = row

    index = shared["scan_source"](source)
    require(not index["invalidReasons"] and not any(index["counts"].get(key, 0)
            for key in ("parse_failure", "identityConflictRows", "invalid")), "SOURCE_CONTRACT_INVALID")
    all_records, summaries, observed_pairs = [], [], set()
    for line, raw, kind in iter_rows(source):
        meta = index["validByLine"].get(line)
        if meta is None or meta["gameId"] not in selected:
            continue
        require(kind == "parsed", "SOURCE_CHANGED_DURING_DIAGNOSTIC")
        _, _, payload, _ = shared["normalize_event"](raw)
        records, summary = diagnose_segment(payload, meta, index["pseudonyms"][meta["gameId"]][meta["playerId"]],
                                             index["problems"].get(line, ()))
        for record in records:
            key = (line, record["inputIndex"])
            record["datasetIncludedPair"] = key in used
            if key in used:
                sample = used[key]
                previous = records[record["inputIndex"] - 1] if record["inputIndex"] else None
                require(previous is not None and sample["previousInputIndex"] == previous["inputIndex"]
                        and sample["recordedIntervalMs"] == record["recordedDeltaMs"]
                        and sample["gameId"] == record["gameId"] and sample["pieceIndex"] == record["pieceIndex"]
                        and sample["playerNumber"] == record["playerNumber"]
                        and sample["previousAction"] == previous["action"]
                        and sample["nextActionDiagnostic"] == record["action"], "DATASET_RAW_PAIR_MISMATCH")
                observed_pairs.add(key)
        all_records.extend(records)
        summaries.append(summary)
    require(set(used) == observed_pairs, "DATASET_PAIR_NOT_FOUND_IN_RAW")
    require(all(sha256_file(path) == expected for path, expected in watched.items()), "INPUT_CHANGED_DURING_DIAGNOSTIC")

    inputs = pd.DataFrame(all_records)
    require(not inputs.empty and target in set(inputs["gameId"]), "TARGET_INPUTS_NOT_FOUND")
    inputs = inputs.sort_values(["gameId", "playerNumber", "sourceLine", "inputIndex"]).reset_index(drop=True)
    group = inputs.groupby(["gameId", "playerNumber"], sort=False)
    inputs["relativeTimeMs"] = inputs["timestampMs"] - group["timestampMs"].transform("first")
    inputs["sourceOrder"] = group.cumcount()
    inputs = inputs.drop(columns="timestampMs")
    segments = pd.DataFrame(summaries)
    comparison = []
    for game in selected:
        raw_inputs = inputs.loc[inputs["gameId"] == game]
        raw_pairs = raw_inputs["recordedDeltaMs"].dropna()
        samples = [row for row in used.values() if row["gameId"] == game]
        comparison.append({"gameId": game, "role": "target" if game == target else "train_comparison",
            "segments": int(segments["gameId"].eq(game).sum()), "inputRecords": len(raw_inputs),
            "rawTimedPairs": len(raw_pairs), "rawZeroPairs": int(raw_pairs.eq(0).sum()),
            "rawZeroPercent": 100 * raw_pairs.eq(0).mean() if len(raw_pairs) else None,
            "datasetPairs": len(samples), "datasetZeroPairs": sum(row["recordedIntervalMs"] == 0 for row in samples),
            "datasetZeroPercent": 100 * sum(row["recordedIntervalMs"] == 0 for row in samples) / len(samples) if samples else None})
    checks = {"scope": "target_plus_up_to_three_context_matched_train_games", "selectedGames": selected,
              "sourceUnchanged": True, "datasetUnchanged": True, "pairLineageMatched": True,
              "modelFitCount": 0, "causeStatus": "unverified", "policyDecisionEligible": False}
    return inputs, segments, pd.DataFrame(comparison), checks


# 5. 실제 진단을 실행하는 셀

이 셀이 raw를 읽습니다. **이번에는 에이전트가 실행하지 않았습니다.**
위 설정·synthetic 예제를 확인한 뒤 사용자가 실행하세요.
checksum 오류가 나면 다음으로 넘어가지 말고 원본·manifest 차이를 확인하세요. 검사를 해제해 계속하지 않습니다.


In [ ]:
inputs, segments, comparison, checks = collect_diagnostic()
target_inputs = inputs.loc[inputs["gameId"] == TARGET_GAME_ID].copy()
target_segments = segments.loc[segments["gameId"] == TARGET_GAME_ID].copy()
print(json.dumps(checks, ensure_ascii=False, indent=2))


# 6. pandas 표로 확인합니다

비교표의 raw는 hold를 포함한 전체 Human 입력, dataset은 이미 채택된 pair만 뜻합니다. 서로 다른 분모를 구분합니다.
0ms 예제에서 sequenceStep=1인지, tickStep이 0/양수/음수인지 봅니다. tick 차이를 ms로 환산하지 않습니다.
동일한 recorded fields가 반복돼도 중복 저장이나 잘못된 입력으로 단정하지 않습니다.
표에서 sourceLine을 골라 `target_inputs.loc[target_inputs["sourceLine"].eq(행번호)]`를 실행하면 해당 segment 전체를 볼 수 있습니다.


In [ ]:
from IPython.display import display

display(comparison)
print("Target segments: longest same-time runs first; a run length counts adjacent pairs, not inputs")
display(target_segments.sort_values(["maxZeroRunPairs", "sourceLine"], ascending=[False, True]).head(30))
columns = ["playerNumber", "pieceIndex", "sourceLine", "inputIndex", "action", "sequence", "tick",
           "relativeTimeMs", "recordedDeltaMs", "sequenceStep", "tickStep", "pieceMatches",
           "zeroRunPairs", "datasetIncludedPair"]
print("Target input examples in source order (first 40 rows)")
display(target_inputs[columns].head(40))
print("Target zero-ms examples (first 30 rows)")
display(target_inputs.loc[target_inputs["recordedDeltaMs"].eq(0), columns].head(30))
print("Order / time issues (not silently removed; first 30 rows)")
issues = (target_inputs["inputIndex"].gt(0) &
          (target_inputs["sequenceStep"].ne(1) | target_inputs["tickStep"].isna() |
           target_inputs["tickStep"].lt(0) | target_inputs["recordedDeltaMs"].isna() |
           target_inputs["recordedDeltaMs"].lt(0))) | ~target_inputs["pieceMatches"]
display(target_inputs.loc[issues, columns].head(30))


# 7. Matplotlib 그래프를 읽는 방법

왼쪽: source 순서상 입력 번호와 첫 유효 관측 입력 대비 상대 시각입니다. player마다 원점이 다르며 게임 시작 이후 반응시간이 아닙니다.
중간: 대상 게임 segment별 0ms 비율입니다. hold·학습 제외 구간도 포함하며 pieceIndex 중복이 있으면 sourceLine으로 구분합니다.
오른쪽: 대상과 동일 문맥 Train 비교 게임의 raw/dataset 0ms 비율입니다. 비교군은 전체 Human 집단의 대표 표본이 아닙니다.
평평한 시각 구간이나 특정 조각의 높은 비율은 **관측 패턴**입니다. 큐 지연·브라우저 정지·timestamp 복사 중 어느 원인인지 자동 판정하지 않습니다.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), layout="constrained")
for player, group in target_inputs.groupby("playerNumber"):
    axes[0].plot(group["sourceOrder"], group["relativeTimeMs"], label=f"Player {player}")
axes[0].set(title="Target: recorded time in source order", xlabel="Input order within player", ylabel="Milliseconds from first valid observation")
axes[0].legend()
for player, group in target_segments.groupby("playerNumber"):
    usable = group.loc[group["validTimedPairs"].gt(0)]
    axes[1].scatter(usable["pieceIndex"], 100 * usable["zeroPairs"] / usable["validTimedPairs"], label=f"Player {player}", alpha=.7)
axes[1].set(title="Target: zero-ms fraction by segment", xlabel="Logged pieceIndex (not elapsed time)", ylabel="Zero-ms pairs (%)", ylim=(-2, 102))
axes[1].legend()
plot_table = comparison.assign(game=comparison["gameId"].str[:8] + " (" + comparison["role"] + ")").set_index("game")
plot_table[["rawZeroPercent", "datasetZeroPercent"]].plot.bar(ax=axes[2], rot=25, color=["#0072B2", "#E69F00"])
axes[2].set(title="Selected games only: different denominators", xlabel="Game / diagnostic role", ylabel="Zero-ms pairs (%)")
fig.suptitle("Human zero-ms diagnostic | read-only | cause unverified")
plt.show()
plt.close(fig)


# 8. 결과를 어떻게 해석할까?

- 순서는 정상인데 시각만 같음: 같은 ms 안의 처리와 일치할 가능성. 실제 동시 키 입력·batch 처리의 증거는 아닙니다.
- 긴 평탄 구간·모든 입력이 placement와 같은 시각: 기록의 해상도·공통 시각 사용을 의심할 단서입니다. 원인 확정에는 당시 recorder/build 증거가 필요합니다.
- sequence/tick/시각 역전: 구조 문제로 기록하고 기존 dataset 제외와의 관계를 확인합니다. 여기서 재분류하지 않습니다.
- 원인 불명: `causeStatus=unverified`로 공유합니다. test 삭제·재분할·clipping 기준 조정을 하지 않습니다.
- placement 시각과 입력 시각의 숫자상 일치도 두 시계의 공통 기원을 입증하지 않습니다.

다음에는 Notebook 비교표·0ms 예제·문제 segment 표를 공유하세요. 학습·새 dataset 생성은 별도 승인 범위입니다.
